# 2.0 Entendimiento de los datos — Saber Pro Genéricas 2024

Este notebook explora el archivo `Examen_Saber_Pro_Genericas_2024.txt` para conocer su tamaño, estructura, calidad, distribuciones y relaciones.

> **Alcance:** este es únicamente un análisis exploratorio. No se rellenan datos faltantes, no se eliminan valores atípicos, no se transforman variables y no se entrenan modelos.

In [ ]:
# Librerías
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Configuración visual
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 110

RANDOM_STATE = 42
PLOT_SAMPLE_SIZE = 60_000

## 1. Carga y validación inicial

Primero se localiza y carga el archivo. Después se revisan sus dimensiones, consumo de memoria y posibles duplicados.

In [ ]:
# La ruta funciona desde la raíz del proyecto y desde notebooks/
path_from_root = Path('data/raw/Examen_Saber_Pro_Genericas_2024.txt')
path_from_notebooks = Path('../data/raw/Examen_Saber_Pro_Genericas_2024.txt')

if path_from_root.exists():
    data_path = path_from_root
else:
    data_path = path_from_notebooks

print('Archivo:', data_path.resolve())
print(f'Tamaño: {data_path.stat().st_size / 1024**2:,.2f} MB')

# El archivo utiliza punto y coma como separador
df = pd.read_csv(data_path, sep=';', encoding='utf-8-sig', low_memory=False)

print(f'Filas: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]:,}')
display(df.head())

# Resumen general
memory_mb = df.memory_usage(deep=True).sum() / 1024**2
duplicate_rows = int(df.duplicated().sum())
duplicate_ids = int(df['estu_consecutivo'].duplicated(keep=False).sum())

overview = pd.DataFrame({
    'Indicador': ['Filas', 'Columnas', 'Memoria en RAM (MB)', 'Filas duplicadas', 'Filas con ID duplicado'],
    'Valor': [len(df), len(df.columns), memory_mb, duplicate_rows, duplicate_ids]
})
display(overview)

## 2. Esquema, tipos y cardinalidad

La cardinalidad es la cantidad de valores diferentes de una columna. Los códigos numéricos se separan de las medidas para no crear histogramas o correlaciones engañosas.

In [ ]:
# Identificadores y códigos
code_columns = []
for column in df.columns:
    is_code = column == 'periodo' or column == 'estu_consecutivo' or '_cod' in column or 'snies' in column
    if is_code:
        code_columns.append(column)

# Medidas numéricas: columnas numéricas que no son códigos
numeric_columns = []
for column in df.select_dtypes(include='number').columns:
    if column not in code_columns:
        numeric_columns.append(column)

# Variables que se describirán como categorías
categorical_columns = []
for column in df.columns:
    if column not in numeric_columns and column != 'estu_fechanacimiento':
        categorical_columns.append(column)

print(f'Medidas numéricas ({len(numeric_columns)}):')
print(numeric_columns)

# Tabla del esquema completo
schema = pd.DataFrame(index=df.columns)
schema['tipo_inferido'] = df.dtypes.astype(str)
schema['datos_presentes'] = df.notna().sum()
schema['datos_faltantes'] = df.isna().sum()
schema['faltantes_pct'] = df.isna().mean() * 100
schema['valores_unicos'] = df.nunique(dropna=True)
schema['cardinalidad_pct'] = df.nunique(dropna=True) / len(df) * 100

display(schema.sort_values('faltantes_pct', ascending=False))

## 3. Calidad y datos faltantes

Pandas representa con `NaN` los campos vacíos. En el archivo original hay 2.988.713 campos vacíos, un valor literal `N/A` y una celda que contiene solo un espacio; esta última no se reconoce automáticamente como faltante.

Se revisan los faltantes por columna, por fila y los campos que suelen faltar simultáneamente.

In [ ]:
# Faltantes por columna
missing_table = pd.DataFrame(index=df.columns)
missing_table['cantidad_faltante'] = df.isna().sum()
missing_table['porcentaje_faltante'] = df.isna().mean() * 100
missing_table = missing_table.sort_values('porcentaje_faltante', ascending=False)

total_missing = int(df.isna().sum().sum())
total_cells = df.shape[0] * df.shape[1]
print(f'Celdas faltantes: {total_missing:,} ({total_missing / total_cells * 100:.2f}%)')
print(f'Columnas con faltantes: {(missing_table.cantidad_faltante > 0).sum()}')
display(missing_table)

# Barras: columnas con más faltantes
top_missing = missing_table[missing_table['cantidad_faltante'] > 0].head(30)
top_missing = top_missing.sort_values('porcentaje_faltante')
plt.figure(figsize=(11, 9))
plt.barh(top_missing.index, top_missing['porcentaje_faltante'], color='steelblue')
plt.axvline(50, color='red', linestyle='--', label='50%')
plt.title('Columnas con mayor porcentaje de datos faltantes')
plt.xlabel('Porcentaje faltante')
plt.legend()
plt.tight_layout()
plt.show()

# Faltantes por fila
missing_per_row = df.isna().sum(axis=1)
display(missing_per_row.describe().to_frame('faltantes_por_fila'))
plt.figure(figsize=(10, 5))
sns.histplot(missing_per_row, discrete=True)
plt.title('Cantidad de datos faltantes por fila')
plt.xlabel('Número de columnas faltantes')
plt.show()

# Mapa del patrón de ausencia en una muestra
missing_sample = df.sample(min(300, len(df)), random_state=RANDOM_STATE).isna()
plt.figure(figsize=(18, 7))
sns.heatmap(missing_sample, cmap=['#23395B', '#F2C14E'], cbar=False, yticklabels=False)
plt.title('Patrón de faltantes: amarillo = faltante, azul = presente')
plt.xticks(rotation=90, fontsize=6)
plt.tight_layout()
plt.show()

In [ ]:
# Columnas que tienen algunos datos faltantes
columns_with_missing = []
for column in df.columns:
    missing_count = df[column].isna().sum()
    if missing_count > 0 and missing_count < len(df):
        columns_with_missing.append(column)

# Valores cercanos a 1: las dos columnas suelen faltar juntas
missing_correlation = df[columns_with_missing].isna().corr()
plt.figure(figsize=(18, 15))
sns.heatmap(missing_correlation, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Mapa de calor de la correlación entre datos faltantes')
plt.xticks(rotation=90, fontsize=6)
plt.yticks(fontsize=6)
plt.tight_layout()
plt.show()

# Fecha de nacimiento: diagnóstico en una serie auxiliar
birth_date = pd.to_datetime(df['estu_fechanacimiento'], dayfirst=True, errors='coerce')
invalid_dates = df['estu_fechanacimiento'].notna().sum() - birth_date.notna().sum()
age_2024 = (pd.Timestamp('2024-12-31') - birth_date).dt.days / 365.25

print('Fechas no interpretables:', invalid_dates)
print('Fecha mínima:', birth_date.min())
print('Fecha máxima:', birth_date.max())
display(age_2024.describe().to_frame('edad_aproximada_en_2024'))

plt.figure(figsize=(10, 5))
sns.histplot(age_2024.dropna(), bins=50)
plt.title('Distribución de la edad aproximada en 2024')
plt.xlabel('Edad')
plt.xlim(age_2024.quantile(0.001), age_2024.quantile(0.999))
plt.show()

## 4. Estadísticas y distribuciones numéricas

Las estadísticas usan todas las filas. Las gráficas usan una muestra reproducible para que el notebook sea más rápido.

In [ ]:
# Estadísticas descriptivas
numeric_stats = df[numeric_columns].describe().T
numeric_stats['mediana'] = df[numeric_columns].median()
numeric_stats['faltantes'] = df[numeric_columns].isna().sum()
numeric_stats['faltantes_pct'] = df[numeric_columns].isna().mean() * 100
numeric_stats['asimetria'] = df[numeric_columns].skew()
display(numeric_stats)

# Muestra usada solamente para las gráficas
plot_sample = df[numeric_columns].sample(min(PLOT_SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE)
columns_per_row = 4
rows_needed = math.ceil(len(numeric_columns) / columns_per_row)

# Histogramas
fig, axes = plt.subplots(rows_needed, columns_per_row, figsize=(18, 3.5 * rows_needed))
axes = np.array(axes).reshape(-1)
for position, column in enumerate(numeric_columns):
    sns.histplot(plot_sample[column].dropna(), bins=35, ax=axes[position])
    axes[position].set_title(column, fontsize=9)
for position in range(len(numeric_columns), len(axes)):
    axes[position].axis('off')
plt.suptitle('Distribuciones de las variables numéricas', y=1.01)
plt.tight_layout()
plt.show()

# Diagramas de caja
fig, axes = plt.subplots(rows_needed, columns_per_row, figsize=(18, 2.7 * rows_needed))
axes = np.array(axes).reshape(-1)
for position, column in enumerate(numeric_columns):
    sns.boxplot(x=plot_sample[column], ax=axes[position], fliersize=1)
    axes[position].set_title(column, fontsize=9)
for position in range(len(numeric_columns), len(axes)):
    axes[position].axis('off')
plt.suptitle('Diagramas de caja de las variables numéricas', y=1.01)
plt.tight_layout()
plt.show()

## 5. Posibles valores atípicos

Se usa la regla IQR: un valor se marca si está por debajo de $Q_1 - 1.5\times IQR$ o por encima de $Q_3 + 1.5\times IQR$. Marcarlo no significa que sea incorrecto. En esta fase no se elimina ni modifica ningún dato.

In [ ]:
# Resumen de posibles atípicos para cada variable numérica
outlier_results = []

for column in numeric_columns:
    values = df[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr
    outlier_count = ((values < lower_limit) | (values > upper_limit)).sum()
    outlier_percentage = outlier_count / len(values) * 100

    outlier_results.append({
        'variable': column,
        'limite_inferior': lower_limit,
        'limite_superior': upper_limit,
        'cantidad_atipicos': outlier_count,
        'porcentaje_atipicos': outlier_percentage
    })

outlier_summary = pd.DataFrame(outlier_results).sort_values('porcentaje_atipicos', ascending=False)
display(outlier_summary)

chart_data = outlier_summary.sort_values('porcentaje_atipicos')
plt.figure(figsize=(11, 8))
plt.barh(chart_data['variable'], chart_data['porcentaje_atipicos'], color='coral')
plt.title('Posibles valores atípicos según la regla IQR')
plt.xlabel('Porcentaje de observaciones no nulas')
plt.tight_layout()
plt.show()

## 6. Variables categóricas

La tabla resume todas las variables categóricas. Las gráficas muestran únicamente variables importantes y sus diez categorías más frecuentes para conservar la legibilidad.

In [ ]:
# Resumen de categorías
categorical_results = []
for column in categorical_columns:
    counts = df[column].value_counts(dropna=True)
    most_common = counts.index[0] if len(counts) > 0 else np.nan
    most_common_count = counts.iloc[0] if len(counts) > 0 else 0
    categorical_results.append({
        'variable': column,
        'categorias_unicas': df[column].nunique(dropna=True),
        'faltantes_pct': df[column].isna().mean() * 100,
        'categoria_mas_frecuente': most_common,
        'frecuencia_categoria_principal': most_common_count
    })

categorical_summary = pd.DataFrame(categorical_results)
display(categorical_summary.sort_values('categorias_unicas', ascending=False))

selected_categories = [
    'estu_genero', 'estu_discapacidad', 'estu_tieneetnia',
    'estu_metodo_prgm', 'estu_nivel_prgm_academico',
    'estu_depto_presentacion', 'inst_caracter_academico',
    'inst_origen', 'fami_estratovivienda', 'fami_educacionmadre',
    'fami_educacionpadre', 'fami_tieneinternet'
]
selected_categories = [column for column in selected_categories if column in df.columns]

fig, axes = plt.subplots(4, 3, figsize=(19, 18))
axes = axes.reshape(-1)
for position, column in enumerate(selected_categories):
    counts = df[column].fillna('<FALTANTE>').value_counts().head(10).sort_values()
    counts.plot(kind='barh', ax=axes[position])
    axes[position].set_title(column, fontsize=9)
    axes[position].set_xlabel('Cantidad de registros')
for position in range(len(selected_categories), len(axes)):
    axes[position].axis('off')
plt.suptitle('Distribuciones de variables categóricas', y=1.01)
plt.tight_layout()
plt.show()

## 7. Correlaciones entre métricas

Pearson describe relaciones lineales y Spearman relaciones que aumentan o disminuyen de manera consistente. Los códigos e identificadores no se incluyen. Una correlación no demuestra causalidad.

In [ ]:
# Correlaciones de Pearson
pearson_correlation = df[numeric_columns].corr(method='pearson')
plt.figure(figsize=(16, 13))
sns.heatmap(pearson_correlation, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Mapa de calor de correlaciones Pearson')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()

# Correlaciones de Spearman
spearman_correlation = df[numeric_columns].corr(method='spearman')
plt.figure(figsize=(16, 13))
sns.heatmap(spearman_correlation, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Mapa de calor de correlaciones Spearman')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()

# Variables más relacionadas con el puntaje global
if 'punt_global' in numeric_columns:
    global_correlations = pd.DataFrame({
        'Pearson': pearson_correlation['punt_global'],
        'Spearman': spearman_correlation['punt_global']
    })
    global_correlations = global_correlations.drop(index='punt_global')
    global_correlations['correlacion_absoluta_maxima'] = global_correlations.abs().max(axis=1)
    display(global_correlations.sort_values('correlacion_absoluta_maxima', ascending=False))

## 8. Relaciones bivariadas exploratorias

Se visualizan las métricas más relacionadas con `punt_global` y su distribución en algunos grupos. Estas son diferencias descriptivas, no efectos causales.

In [ ]:
# Seis métricas con mayor correlación de Spearman con punt_global
correlations_with_global = spearman_correlation['punt_global'].drop('punt_global').abs()
top_six = correlations_with_global.sort_values(ascending=False).head(6).index
scatter_sample = df[list(top_six) + ['punt_global']].sample(min(15_000, len(df)), random_state=RANDOM_STATE)

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.reshape(-1)
for position, column in enumerate(top_six):
    sns.scatterplot(data=scatter_sample, x=column, y='punt_global', alpha=0.2, s=10, ax=axes[position])
    axes[position].set_title(f'punt_global vs. {column}', fontsize=9)
plt.tight_layout()
plt.show()

# Puntaje global por algunos grupos
group_columns = ['estu_genero', 'estu_metodo_prgm', 'estu_nivel_prgm_academico', 'inst_origen', 'inst_caracter_academico', 'fami_estratovivienda']
fig, axes = plt.subplots(3, 2, figsize=(17, 15))
axes = axes.reshape(-1)
for position, column in enumerate(group_columns):
    most_common_groups = df[column].value_counts().head(10).index
    group_data = df[df[column].isin(most_common_groups)][[column, 'punt_global']].dropna()
    if len(group_data) > PLOT_SAMPLE_SIZE:
        group_data = group_data.sample(PLOT_SAMPLE_SIZE, random_state=RANDOM_STATE)
    sns.boxplot(data=group_data, y=column, x='punt_global', showfliers=False, ax=axes[position])
    axes[position].set_title(f'punt_global por {column}', fontsize=9)
plt.tight_layout()
plt.show()

## 9. Resumen de alertas de calidad

Esta tabla reúne señales que merecen revisión posterior. No realiza ninguna corrección.

In [ ]:
quality_summary = pd.DataFrame({
    'Indicador': [
        'Columnas totalmente vacías',
        'Columnas con más de 50% faltante',
        'Columnas constantes',
        'Columnas con cardinalidad mayor a 90%',
        'Filas duplicadas',
        'Filas con identificador duplicado',
        'Variables numéricas con posibles atípicos'
    ],
    'Cantidad': [
        int((schema['faltantes_pct'] == 100).sum()),
        int((schema['faltantes_pct'] > 50).sum()),
        int((schema['valores_unicos'] == 1).sum()),
        int((schema['cardinalidad_pct'] > 90).sum()),
        duplicate_rows,
        duplicate_ids,
        int((outlier_summary['cantidad_atipicos'] > 0).sum())
    ]
})
display(quality_summary)

## 10. Guía para interpretar los resultados

Al ejecutar el notebook, conviene concentrarse en:

1. El número de estudiantes, instituciones, programas y territorios representados.
2. Las columnas vacías o con muchos faltantes.
3. Las distribuciones muy asimétricas o concentradas.
4. Los posibles atípicos, recordando que no necesariamente son errores.
5. Las correlaciones fuertes y las diferencias entre Pearson y Spearman.
6. Las diferencias descriptivas entre grupos, sin interpretarlas como causalidad.

> Las decisiones de limpieza o preparación deberán justificarse posteriormente. Este notebook no modifica los datos.